# Simple real NISAR GUNW workflow

Set `SNOWIN_GUNW`, `SNOWIN_NISAR_DEM`, and `SNOWIN_DEM_SOURCE` in the environment or ignored repository-root `.env` file. This full-scene workflow inspects the normalized phase and opened GUNW layers, adds local-incidence geometry using the declared DEM source, and previews pairwise dSWE. Orthometric DEM inputs without a vertical correction are marked provisional. The correction layers are visualized for inspection and are not applied automatically. For a bounded local geometry check, run `07_local_gunw_dem_smoke.ipynb`.

In [ ]:
import os
from html import escape
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from IPython.display import HTML, display

from snowin import compute_dswe
from snowin.io import add_gunw_incidence, open_gunw


ROOT_CANDIDATES = (Path.cwd(), *Path.cwd().parents)
REPO_ROOT = next(
    (candidate for candidate in ROOT_CANDIDATES if (candidate / "pyproject.toml").is_file()),
    Path.cwd(),
)


def load_local_env():
    env_path = REPO_ROOT / ".env"
    if not env_path.is_file():
        return
    allowed_keys = {"SNOWIN_GUNW", "SNOWIN_NISAR_DEM", "SNOWIN_DEM_SOURCE"}
    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        key, separator, value = line.partition("=")
        key = key.strip()
        if separator and key in allowed_keys:
            os.environ.setdefault(key, value.strip().strip("\"'"))


load_local_env()


def local_path(variable, prompt):
    value = os.environ.get(variable, "").strip()
    if not value:
        value = input(prompt).strip()
    if not value:
        raise RuntimeError(f"A value for {variable} is required.")
    path = Path(value).expanduser()
    if not path.is_absolute():
        local_candidates = (Path.cwd() / path, REPO_ROOT / path)
        path = next((candidate for candidate in local_candidates if candidate.is_file()), local_candidates[-1])
    path = path.resolve()
    if not path.is_file():
        raise FileNotFoundError(
            f"{variable} does not name an existing file: {path}. "
            "Use an absolute path or a repository-relative path; a bare basename "
            "is not enough unless it is in the notebook's current directory."
        )
    return path


GUNW_PATH = local_path("SNOWIN_GUNW", "Enter the path to a local NISAR GUNW product: ")
DEM_PATH = local_path("SNOWIN_NISAR_DEM", "Enter the path to a prepared local DEM: ")
dem_source = os.environ.get("SNOWIN_DEM_SOURCE", "").strip().lower()
if not dem_source:
    dem_source = input("DEM source (nisar_cop30, cop30, tandem30, srtm30): ").strip().lower()
if dem_source not in {"nisar_cop30", "cop30", "tandem30", "srtm30"}:
    raise ValueError(
        "Set SNOWIN_DEM_SOURCE to nisar_cop30, cop30, tandem30, or srtm30."
    )

with rasterio.open(DEM_PATH) as _dem:
    if _dem.crs is None:
        raise ValueError("The prepared DEM must declare a coordinate reference system.")
    dem_tags = _dem.tags()
    dem_height_reference = dem_tags.get("height_reference", "").strip().lower()
    dem_summary = {
        "crs": _dem.crs.to_string(),
        "shape_y_x": (_dem.height, _dem.width),
        "source_product": dem_tags.get("source_product"),
        "height_reference_tag": dem_tags.get("height_reference"),
    }
expected_height_references = {
    "nisar_cop30": {"ellipsoid", "ellipsoidal", "wgs84 ellipsoid"},
    "cop30": {"orthometric", "egm2008"},
    "tandem30": {"ellipsoid", "ellipsoidal", "wgs84 ellipsoid", "wgs84-g1150"},
    "srtm30": {"orthometric", "egm96"},
}
if dem_height_reference and dem_height_reference not in expected_height_references[dem_source]:
    raise ValueError(
        f"DEM height_reference tag {dem_height_reference!r} conflicts with "
        f"SNOWIN_DEM_SOURCE={dem_source!r}."
    )

print(f"GUNW: {GUNW_PATH.name}")
print(f"DEM:  {DEM_PATH.name}")
print(f"DEM source declaration: {dem_source}")
print(f"DEM metadata: {dem_summary}")
if not dem_height_reference:
    print("The DEM has no height_reference tag; confirm its source declaration from local provenance.")
if dem_source in {"cop30", "srtm30"}:
    print("Orthometric DEM; no vertical correction is configured, so geometry is provisional.")


def _decode_hdf5_value(value):
    if isinstance(value, (bytes, np.bytes_)):
        return bytes(value).decode("utf-8", errors="replace")
    if isinstance(value, np.ndarray):
        if value.shape == ():
            return _decode_hdf5_value(value.item())
        return [_decode_hdf5_value(item) for item in value.flat]
    if isinstance(value, np.generic):
        return value.item()
    return value


def _format_metadata_value(value):
    if isinstance(value, (float, np.floating)):
        return f"{float(value):.6g}"
    return str(value)


def _display_metadata(title, attrs, fields):
    rows = "".join(
        f"<tr><th><code>{escape(field)}</code></th><td>{escape(_format_metadata_value(attrs.get(field, 'not present')))}</td></tr>"
        for field in fields
    )
    display(HTML(f"<h3>{escape(title)}</h3><table><thead><tr><th>Field</th><th>Value</th></tr></thead><tbody>{rows}</tbody></table>"))

## Inspect product metadata

In [ ]:
phase_path = (
    "science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram/HH/unwrappedPhase"
)
height_path = "science/LSAR/GUNW/metadata/radarGrid/heightAboveEllipsoid"
reference_path = "science/LSAR/identification/referenceZeroDopplerStartTime"
secondary_path = "science/LSAR/identification/secondaryZeroDopplerStartTime"

with h5py.File(GUNW_PATH, "r") as _h5:
    phase_attrs = {key: _decode_hdf5_value(value) for key, value in _h5[phase_path].attrs.items()}
    height_attrs = {key: _decode_hdf5_value(value) for key, value in _h5[height_path].attrs.items()}
_display_metadata("Phase layer metadata", phase_attrs, ["units", "description", "grid_mapping", "min_value", "max_value", "mean_value", "sample_stddev"])
_display_metadata("Radar-grid height metadata", height_attrs, ["standard_name", "description", "units"])
with h5py.File(GUNW_PATH, "r") as _h5:
    reference_time = _decode_hdf5_value(_h5[reference_path][()])
    secondary_time = _decode_hdf5_value(_h5[secondary_path][()])
display(HTML(f"<h3>Acquisition times</h3><ul><li><b>Reference</b>: <code>{escape(str(reference_time))}</code></li><li><b>Secondary</b>: <code>{escape(str(secondary_time))}</code></li></ul>"))

## Open and inspect the GUNW with SnowIn

This step does not download a DEM or compute incidence. Phase/product arrays remain lazy when Dask is installed, so you can inspect metadata before starting the slower geometry step.

In [ ]:
pair = open_gunw(GUNW_PATH, chunks="auto")
display(pair)
display(HTML(
    f"<p><b>Phase transform</b>: <code>{pair.attrs.get('phase_transform')}</code><br><b>Incidence status</b>: <code>{pair.attrs.get('incidence_angle_status')}</code><br><b>Wavelength</b>: <code>{pair.attrs.get('wavelength_m'):.9f} m</code></p>"
))

## Add local incidence when ready

This explicit call requires the GUNW path and appends `incidence_angle` and `geometry_valid` to the already-open dataset. It reprojects the caller-supplied prepared DEM according to `SNOWIN_DEM_SOURCE` and reads the GUNW LOS vectors.

In [ ]:
add_gunw_incidence(
    pair, GUNW_PATH, dem=DEM_PATH, dem_source=dem_source,
    require_vertical_datum_match=dem_source in {"nisar_cop30", "tandem30"},
    progress=True
)
display(pair[["incidence_angle", "geometry_valid"]])
geometry_info = {
    key: pair.attrs.get(key)
    for key in ("incidence_angle_source", "incidence_angle_reference", "dem_source", "dem_height_reference", "vertical_datum_status", "geometry_execution")
}
display(HTML(f"<h3>Geometry provenance</h3><pre>{escape(str(geometry_info))}</pre>"))

## Preview the opened layers, local incidence, and pairwise dSWE

This plot includes every layer returned by `open_gunw`, including available correction screens, plus the computed incidence and dSWE. Local incidence is displayed in degrees; SnowIn calculations use radians.

In [ ]:
step = max(1, min(pair.sizes["y"], pair.sizes["x"]) // 600)
safe_incidence = pair["incidence_angle"].where(pair["geometry_valid"])
dswe = compute_dswe(
    pair["phase"],
    safe_incidence,
    wavelength_m=float(pair.attrs["wavelength_m"]),
)
display(dswe)
phase_preview = pair["phase"].isel(y=slice(None, None, step), x=slice(None, None, step))
dswe_preview = dswe.isel(y=slice(None, None, step), x=slice(None, None, step))

plot_layers = list(pair.data_vars) + ["dswe"]
ncols = 3
nrows = max(1, (len(plot_layers) + ncols - 1) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.8 * nrows), squeeze=False)

for ax, name in zip(axes.flat, plot_layers):
    layer = dswe if name == "dswe" else pair[name]
    title = name.replace("_", " ").title()
    if "radar_height" in layer.dims:
        layer = layer.sel(radar_height=0, method="nearest")
        title += f" (radar height {float(layer.radar_height.values):g} m)"

    y_dim = "radar_y" if "radar_y" in layer.dims else "y"
    x_dim = "radar_x" if "radar_x" in layer.dims else "x"
    if name == "connected_component" or name == "geometry_valid":
        preview = layer.isel({y_dim: slice(None, None, step), x_dim: slice(None, None, step)})
        cmap = "tab20" if name == "connected_component" else "Greens"
    else:
        preview = layer.coarsen({y_dim: step, x_dim: step}, boundary="trim").mean()
        cmap = {
            "phase": "RdBu_r",
            "coherence": "plasma",
            "incidence_angle": "viridis",
            "dswe": "BrBG",
            "ionosphere": "RdBu_r",
            "ionosphere_unc": "magma",
            "hydro_tropo": "RdBu_r",
            "wet_tropo": "RdBu_r",
        }.get(name, "magma")
    kwargs = {"vmin": 0, "vmax": 1} if name == "geometry_valid" else {"robust": True}
    preview.compute().plot.imshow(ax=ax, cmap=cmap, **kwargs)
    ax.set_title(title)

for ax in axes.flat[len(plot_layers):]:
    fig.delaxes(ax)

fig.suptitle("Opened GUNW layers, local incidence, and pairwise dSWE")
fig.tight_layout()
plt.show()

In [ ]:
display(HTML(
f"<h3>Output checks</h3><table><tr><th>Check</th><th>Value</th></tr><tr><td>Phase definition</td><td><code>{pair.attrs.get("phase_difference_definition")}</code></td></tr><tr><td>Incidence reference</td><td><code>{pair["incidence_angle"].attrs.get("incidence_angle_reference")}</code></td></tr><tr><td>Geometry support fraction</td><td>{float(pair.geometry_valid.mean()):.4f}</td></tr><tr><td>Correction layers applied</td><td>{pair.attrs.get("correction_layers_applied", False)}</td></tr><tr><td>Finite phase fraction</td><td>{float(np.isfinite(phase_preview).mean()):.4f}</td></tr><tr><td>Finite dSWE fraction</td><td>{float(np.isfinite(dswe_preview).mean()):.4f}</td></tr></table>"
))
pair.close()